In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

project_dir = "/content/drive/MyDrive/TISER_Project"
data_dir = f"{project_dir}/data"
results_dir = f"{project_dir}/results"
figures_dir = f"{project_dir}/figures"
notebooks_dir = f"{project_dir}/notebooks"

for d in [project_dir, data_dir, results_dir, figures_dir, notebooks_dir]:
    os.makedirs(d, exist_ok=True)

print("Project folders ready")
print(project_dir)

Project folders ready
/content/drive/MyDrive/TISER_Project


In [ ]:
import json
import os

def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"Saved: {path}")


def load_json(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        print(f"Loaded existing file: {path}")
        print("Existing records:", len(data))
        return data

    print("No existing file found. Starting fresh.")
    return []

In [ ]:

# Experiment Configuration
# ================================

EXPERIMENT_NAME = "qwen_baseline_100"

RESULTS_FILE = f"{results_dir}/{EXPERIMENT_NAME}.json"

print("Experiment:", EXPERIMENT_NAME)
print("Results file:", RESULTS_FILE)

Experiment: qwen_baseline_100
Results file: /content/drive/MyDrive/TISER_Project/results/qwen_baseline_100.json


In [ ]:
# Load existing results if they exist
baseline_results_100 = load_json(RESULTS_FILE)

# Resume from where we stopped
start_idx = len(baseline_results_100)

print("Already processed:", start_idx)
print("Will continue from sample:", start_idx)

Loaded existing file: /content/drive/MyDrive/TISER_Project/results/qwen_baseline_100.json
Existing records: 100
Already processed: 100
Will continue from sample: 100


In [ ]:
import zipfile
import json

test_examples = []

test_zip_path = f"{data_dir}/TISER_test.zip"

with zipfile.ZipFile(test_zip_path, "r") as z:
    with z.open("TISER_test.json") as f:
        for line in f:
            test_examples.append(
                json.loads(line.decode("utf-8"))
            )

print("Loaded test examples:", len(test_examples))

Loaded test examples: 22014


In [ ]:
!pip install -q transformers peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.5 MB/s eta 0:00:00


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

base_model_id = "Qwen/Qwen2.5-3B-Instruct"
adapter_id = "arianmo47/de_it_fr_en_mixed_tiser_full_15000_8gb_val_qlora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    adapter_id
)

model.eval()

print("Fine-tuned Qwen TISER model loaded successfully")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/120M [00:00<?, ?B/s]

Fine-tuned Qwen TISER model loaded successfully


In [ ]:
import re

def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*,\s*", ", ", text)
    text = text.rstrip(".")
    return text


def extract_answer(response):
    match = re.search(
        r"<answer>\s*(.*?)\s*</answer>",
        response,
        re.DOTALL | re.IGNORECASE
    )

    if match:
        return match.group(1).strip()

    before_reasoning = response.split("<reasoning>")[0].strip()

    if before_reasoning:
        return before_reasoning.split("\n")[0].strip()

    return "NO_ANSWER_FOUND"

print("Utility functions ready")

Utility functions ready


In [ ]:
# Reload existing saved results, if any
baseline_results_100 = load_json(RESULTS_FILE)

start_idx = len(baseline_results_100)

print("Starting from index:", start_idx)

for idx in range(start_idx, 100):

    sample = test_examples[idx]
    prompt = sample["prompt"]

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=400
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    prediction = extract_answer(response)

    baseline_results_100.append({
        "index": idx,
        "question": sample["question"],
        "ground_truth": sample["answer"],
        "prediction": prediction,
        "full_response": response
    })

    if (idx + 1) % 10 == 0:
        save_json(baseline_results_100, RESULTS_FILE)
        print(f"Processed and saved {idx + 1}/100")

# Final save
save_json(baseline_results_100, RESULTS_FILE)

print("\nBaseline evaluation complete")
print("Total results:", len(baseline_results_100))

Loaded existing file: /content/drive/MyDrive/TISER_Project/results/qwen_baseline_100.json
Existing records: 100
Starting from index: 100
Saved: /content/drive/MyDrive/TISER_Project/results/qwen_baseline_100.json

Baseline evaluation complete
Total results: 100


In [ ]:
correct = 0

for r in baseline_results_100:
    pred = normalize_text(r["prediction"])
    gt = normalize_text(r["ground_truth"])

    if pred == gt:
        correct += 1

print("Correct:", correct)
print("Incorrect:", len(baseline_results_100) - correct)
print("Accuracy:", correct / len(baseline_results_100))

Correct: 49
Incorrect: 51
Accuracy: 0.49


In [ ]:
print(len(baseline_results_100))

print(baseline_results_100[0].keys())

100
dict_keys(['index', 'question', 'ground_truth', 'prediction', 'full_response'])


In [ ]:
print(baseline_results_100[0])

{'index': 0, 'question': 'Which event started first, (Chris Evans was married to Oliver Jensen) or (Chris Evans was born in Bristol, Connecticut)?', 'ground_truth': 'Chris Evans was born in Bristol, Connecticut', 'prediction': 'Chris Evans was born in Bristol,Connecticut', 'full_response': ' Chris Evans was born in Bristol,Connecticut\n\n<reasoning> \nTo determine which event started first, we need to compare the starting times of the two events in question: (Chris Evans was born in Bristol, Connecticut) and (Chris Evans was married to Oliver Jensen). According to the provided temporal context, (Chris Evans was born in Bristol, Connecticut) starts at 1948, while (Chris Evans was married to Oliver Jensen) starts at 1970. Since 1948 is earlier than 1970, it is clear that (Chris Evans was born in Bristol, Connecticut) started first.\n<timeline> \n(Chris Evans was born in Bristol, Connecticut) starts at 1948.\n(Chris Evans was married to Oliver Jensen) starts at 1970.\n</timeline> \n<refle

In [ ]:
from collections import defaultdict

def get_question_type(question):

    q = question.lower()

    if "started first" in q:
        return "started_first"

    elif "chronological order" in q:
        return "chronological_order"

    elif "how long did" in q:
        return "duration"

    elif "how much time passed" in q:
        return "time_difference"

    elif "when did" in q:
        return "timestamp"

    else:
        return "other"


stats = defaultdict(lambda: {"correct": 0, "total": 0})

for r in baseline_results_100:

    qtype = get_question_type(r["question"])

    stats[qtype]["total"] += 1

    if normalize_text(r["prediction"]) == normalize_text(r["ground_truth"]):
        stats[qtype]["correct"] += 1


for qtype, s in stats.items():

    acc = s["correct"] / s["total"]

    print(f"{qtype}: {s['correct']}/{s['total']} = {acc:.2%}")

started_first: 9/9 = 100.00%
chronological_order: 11/30 = 36.67%
duration: 3/9 = 33.33%
time_difference: 0/9 = 0.00%
timestamp: 9/9 = 100.00%
other: 17/34 = 50.00%


In [ ]:
for r in baseline_results_100:
    qtype = get_question_type(r["question"])

    if qtype == "other":
        print("=" * 80)
        print(r["question"])

True or false: event (Oliver Jensen was married to Chris Evans) and event (Oliver Jensen was born in Bristol, Connecticut) started at the same year?
True or false: event (Oliver Jensen was married to Chris Evans) and event (Chris Evans owned Pearl Network) started at the same year?
True or false: event (Chris Evans was married to Oliver Jensen) and event (Chris Evans owned Pearl Network) started at the same year?
True or false: event (Oliver Jensen was married to Chris Evans) was still happening when event (Chris Evans won prize Victory Achievement Award) started?
True or false: event (Oliver Jensen was married to Chris Evans) was still happening when event (Chris Evans owned Pearl Network) started?
What happened right after the event (Chris Evans owned Pearl Network) starts?
What happened right after the event (Oliver Jensen was born in Bristol, Connecticut) starts?
What happened right before the event (Chris Evans owned Pearl Network) ends?
What happened right before the event (Chris

In [ ]:
no_answer_cases = [
    r for r in baseline_results_100
    if r["prediction"] == "NO_ANSWER_FOUND"
]

print("NO_ANSWER_FOUND cases:", len(no_answer_cases))

for r in no_answer_cases[:5]:
    print("=" * 80)
    print("INDEX:", r["index"])
    print("QUESTION:")
    print(r["question"])
    print("\nGROUND TRUTH:")
    print(r["ground_truth"])
    print("\nFULL RESPONSE:")
    print(r["full_response"][:2000])

NO_ANSWER_FOUND cases: 0


In [ ]:
wrong_cases = []

for r in baseline_results_100:
    pred = normalize_text(r["prediction"])
    gt = normalize_text(r["ground_truth"])

    if pred != gt:
        wrong_cases.append(r)

print("Wrong cases:", len(wrong_cases))

for r in wrong_cases[:10]:
    print("=" * 80)
    print("INDEX:", r["index"])
    print("QUESTION:")
    print(r["question"])
    print("\nGROUND TRUTH:")
    print(r["ground_truth"])
    print("\nPREDICTION:")
    print(r["prediction"])

Wrong cases: 51
INDEX: 4
QUESTION:
Given the following five events: (Chris Evans won prize Victory Achievement Award), (Chris Evans owned Pearl Network), (Chris Evans was born in Bristol, Connecticut), (Chris Evans was married to Oliver Jensen), (Oliver Jensen was born in Bristol, Connecticut). Which event is the second one in chronological order?

GROUND TRUTH:
Chris Evans was born in Bristol, Connecticut

PREDICTION:
Chris Evans owned Pearl Network
INDEX: 6
QUESTION:
Given the following five events: (Chris Evans won prize Victory Achievement Award), (Chris Evans owned Pearl Network), (Chris Evans was born in Bristol, Connecticut), (Chris Evans was married to Oliver Jensen), (Oliver Jensen was born in Bristol, Connecticut). Which event is the fourth one in chronological order?

GROUND TRUTH:
Chris Evans owned Pearl Network

PREDICTION:
Oliver Jensen was married to Chris Evans
INDEX: 7
QUESTION:
Given the following five events: (Chris Evans won prize Victory Achievement Award), (Chris 

In [ ]:
import re

def normalize_for_eval(text):
    text = normalize_text(text)

    # Convert "40 years" -> "40"
    # Convert "1 year" -> "1"
    match = re.fullmatch(r"(\d+)\s*years?", text)
    if match:
        return match.group(1)

    return text

In [ ]:
correct = 0

for r in baseline_results_100:
    pred = normalize_for_eval(r["prediction"])
    gt = normalize_for_eval(r["ground_truth"])

    if pred == gt:
        correct += 1

print("Correct:", correct)
print("Incorrect:", len(baseline_results_100) - correct)
print("Accuracy:", correct / len(baseline_results_100))

Correct: 59
Incorrect: 41
Accuracy: 0.59


# Experiment B: Qwen Baseline with Repo-Style Inference

In [ ]:
EXPERIMENT_NAME = "qwen_baseline_100_repo_style"
RESULTS_FILE = f"{results_dir}/{EXPERIMENT_NAME}.json"

qwen_repo_style_100 = load_json(RESULTS_FILE)

print("Experiment:", EXPERIMENT_NAME)
print("Already processed:", len(qwen_repo_style_100))

Loaded existing file: /content/drive/MyDrive/TISER_Project/results/qwen_baseline_100_repo_style.json
Existing records: 100
Experiment: qwen_baseline_100_repo_style
Already processed: 100


In [ ]:
start_idx = len(qwen_repo_style_100)

for idx in range(start_idx, 100):

    sample = test_examples[idx]
    prompt = sample["prompt"]

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.1
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    prediction = extract_answer(response)

    qwen_repo_style_100.append({
        "index": idx,
        "question": sample["question"],
        "ground_truth": sample["answer"],
        "prediction": prediction,
        "full_response": response
    })

    save_json(qwen_repo_style_100, RESULTS_FILE)

    print(f"Processed and saved {idx+1}/100")

print("\nRepo-style baseline evaluation complete")
print("Total results:", len(qwen_repo_style_100))


Repo-style baseline evaluation complete
Total results: 100


In [ ]:
correct = 0

for r in qwen_repo_style_100:
    pred = normalize_for_eval(r["prediction"])
    gt = normalize_for_eval(r["ground_truth"])

    if pred == gt:
        correct += 1

print("Correct:", correct)
print("Incorrect:", len(qwen_repo_style_100) - correct)
print("Accuracy:", correct / len(qwen_repo_style_100))

Correct: 59
Incorrect: 41
Accuracy: 0.59


In [ ]:
import re
from collections import Counter

def tokenize_for_f1(text):
    text = normalize_text(text)
    return re.findall(r"\w+", text)

def f1_score(prediction, ground_truth):
    pred_tokens = tokenize_for_f1(prediction)
    gt_tokens = tokenize_for_f1(ground_truth)

    if len(pred_tokens) == 0 and len(gt_tokens) == 0:
        return 1.0

    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return 0.0

    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)

    return 2 * precision * recall / (precision + recall)

In [ ]:
f1_scores = []

for r in qwen_repo_style_100:
    f1_scores.append(
        f1_score(
            r["prediction"],
            r["ground_truth"]
        )
    )

print("Average F1:", sum(f1_scores) / len(f1_scores))

Average F1: 0.7431600948071536


In [ ]:
import re
from collections import Counter

def repo_normalize(text):
    text = str(text).lower()

    # remove common tags
    text = re.sub(r"</?answer>", " ", text)
    text = re.sub(r"</?reasoning>", " ", text)
    text = re.sub(r"</?timeline>", " ", text)
    text = re.sub(r"</?reflection>", " ", text)

    # punctuation -> spaces
    text = re.sub(r"[^a-z0-9]+", " ", text)

    # collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def repo_f1(prediction, ground_truth):
    pred = repo_normalize(prediction)
    gt = repo_normalize(ground_truth)

    pred_tokens = pred.split()
    gt_tokens = gt.split()

    if len(pred_tokens) == 0 and len(gt_tokens) == 0:
        return 1.0

    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return 0.0

    common = Counter(pred_tokens) & Counter(gt_tokens)
    overlap = sum(common.values())

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(gt_tokens)

    return 2 * precision * recall / (precision + recall)


f1_scores = []

for r in qwen_repo_style_100:
    f1_scores.append(
        repo_f1(
            r["prediction"],
            r["ground_truth"]
        )
    )

print("Repo-style F1:", sum(f1_scores) / len(f1_scores))

Repo-style F1: 0.7431600948071536


In [ ]:
!git clone https://github.com/arianmo477/MultilingualTemporalReasoningTISER.git

Cloning into 'MultilingualTemporalReasoningTISER'...
remote: Enumerating objects: 348, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 348 (delta 40), reused 63 (delta 36), pack-reused 263 (from 3)
Receiving objects: 100% (348/348), 110.01 MiB | 16.02 MiB/s, done.
Resolving deltas: 100% (92/92), done.
Updating files: 100% (96/96), done.
Filtering content: 100% (47/47), 1.94 GiB | 33.26 MiB/s, done.


In [ ]:
%cd MultilingualTemporalReasoningTISER

!find . -maxdepth 3 -type f

/content/MultilingualTemporalReasoningTISER
./LICENSE
./environment.yml
./CODE_OF_CONDUCT.md
./data/prompts/tiser_full_de.txt
./data/prompts/tiser_full_fr.txt
./data/prompts/tiser_full_it.txt
./data/prompts/tiser_full_en.txt
./data/.gitattributes
./data/invalid_samples_train.json
./data/TISER_test.json
./data/invalid_samples_test.json
./data/TISER_train.json
./CONTRIBUTING.md
./multilingual_rag_tiser/rag/retriever.py
./multilingual_rag_tiser/rag/__init__.py
./multilingual_rag_tiser/rag/rag_prompt.py
./multilingual_rag_tiser/rag/build_rag_index.py
./multilingual_rag_tiser/evaluation/inference.py
./multilingual_rag_tiser/evaluation/run_eval_rag.sh
./multilingual_rag_tiser/preprocess/validate_tiser_dataset.py
./multilingual_rag_tiser/preprocess/run_validate_dataset.sh
./multilingual_rag_tiser/preprocess/run_mix_dataset.sh
./multilingual_rag_tiser/preprocess/mix_dataset.py
./multilingual_rag_tiser/training/run_train.sh
./multilingual_rag_tiser/training/train_qlora.py
./multilingual_rag_tis

In [ ]:
!sed -n '1,220p' README.md

# tiser-multilingual

Multilingual temporal reasoning with a small language model — a two-part extension of [TISER](https://aclanthology.org/2025.acl-long.1358/) (Bazaga et al., ACL 2025):

1. **Small-model adaptation** — Qwen2.5-3B-Instruct + QLoRA on a single 8 GB GPU, matching the spirit of the original paper's reasoning pipeline without requiring 40+ GB of VRAM.

2. **Multilingual extension** — an NLLB-based translation pipeline that produces structurally faithful Italian, German, French, and Persian training data, enabling a single fine-tuned model to reason temporally across languages.

The original TISER paper uses Qwen2.5-7B and Mistral-7B on English only. This project reaches strong English-only performance with a 3 B model and extends the same temporal-reasoning pipeline to multilingual EN · IT · DE · FR training.

---

## Results

Most evaluations use approximately **500 samples** across the five TISER categories: `tgqa`, `tempreason_l2`, `tempreason_l3`, `timeqa_easy`, and 

In [ ]:
from collections import Counter

dataset_counts = Counter(
    sample["dataset_name"]
    for sample in test_examples
)

print(dataset_counts)

Counter({'tempreason_l2_test': 5397, 'tempreason_l3_test': 4426, 'tgqa_test': 3316, 'timeqa_hard_test': 3078, 'timeqa_easy_test': 2997, 'tot_semantic_test': 2800})


In [ ]:
target_datasets = [
    "tgqa_test",
    "tempreason_l2_test",
    "tempreason_l3_test",
    "timeqa_easy_test",
    "timeqa_hard_test"
]

balanced_500 = []

for dataset_name in target_datasets:
    selected = [
        sample for sample in test_examples
        if sample["dataset_name"] == dataset_name
    ][:100]

    print(dataset_name, len(selected))
    balanced_500.extend(selected)

print("Total balanced samples:", len(balanced_500))

tgqa_test 100
tempreason_l2_test 100
tempreason_l3_test 100
timeqa_easy_test 100
timeqa_hard_test 100
Total balanced samples: 500


In [ ]:
EXPERIMENT_NAME = "qwen_baseline_500_balanced_repo_style"
RESULTS_FILE = f"{results_dir}/{EXPERIMENT_NAME}.json"

qwen_baseline_500 = load_json(RESULTS_FILE)

print("Experiment:", EXPERIMENT_NAME)
print("Already processed:", len(qwen_baseline_500))

Loaded existing file: /content/drive/MyDrive/TISER_Project/results/qwen_baseline_500_balanced_repo_style.json
Existing records: 500
Experiment: qwen_baseline_500_balanced_repo_style
Already processed: 500


In [ ]:
start_idx = len(qwen_baseline_500)

for idx in range(start_idx, 500):

    sample = balanced_500[idx]
    prompt = sample["prompt"]

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.1
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    prediction = extract_answer(response)

    qwen_baseline_500.append({
        "index": idx,
        "dataset_name": sample["dataset_name"],
        "question": sample["question"],
        "ground_truth": sample["answer"],
        "prediction": prediction,
        "full_response": response
    })

    save_json(qwen_baseline_500, RESULTS_FILE)

    print(f"Processed and saved {idx+1}/500")

print("\nBalanced 500 baseline evaluation complete")
print("Total results:", len(qwen_baseline_500))


Balanced 500 baseline evaluation complete
Total results: 500


In [ ]:
correct = 0

for r in qwen_baseline_500:
    pred = normalize_for_eval(r["prediction"])
    gt = normalize_for_eval(r["ground_truth"])

    if pred == gt:
        correct += 1

print("NormEM Correct:", correct)
print("Total:", len(qwen_baseline_500))
print("NormEM:", correct / len(qwen_baseline_500))

NormEM Correct: 305
Total: 500
NormEM: 0.61


In [ ]:
f1_scores = []

for r in qwen_baseline_500:
    f1_scores.append(
        repo_f1(
            r["prediction"],
            r["ground_truth"]
        )
    )

print("F1:", sum(f1_scores) / len(f1_scores))

F1: 0.7033636262430379


In [ ]:
# Show repo prompt
with open("data/prompts/tiser_full_en.txt", "r", encoding="utf-8") as f:
    repo_prompt = f.read()

print("Repo prompt first 1000 chars:")
print(repo_prompt[:1000])

print("\n" + "="*80 + "\n")

# Show one dataset prompt
sample_prompt = balanced_500[0]["prompt"]

print("Dataset prompt first 1000 chars:")
print(sample_prompt[:1000])

Repo prompt first 1000 chars:
You are an AI assistant that uses a Chain of Thought (CoT) approach with reflection to answer queries. Follow the steps below:

Step 1: Reason through the problem step by step within the <reasoning> section.

Step 2: Based on your reasoning, identify the relevant temporal events needed to answer the given question within the <timeline> section. Assume that temporal relations in the context are unidirectional.

Step 3: Review your reasoning and the constructed timeline to check for any errors or possible improvements within the <reflection> section.

Step 4: If necessary, revise your reasoning based on your reflection. If additional reasoning is required, return to Step 1; otherwise, proceed to the next step.

Step 5: Provide the final, concise answer only within the <answer> section. If the answer is a number, output only the number and nothing else. Otherwise, output only the entity or event name without any additional explanation.

Additional instruction

In [ ]:
print("Repo prompt length:", len(repo_prompt))
print("Dataset prompt length:", len(sample_prompt))

print("Repo prompt in dataset prompt?", repo_prompt.strip()[:200] in sample_prompt)

Repo prompt length: 1577
Dataset prompt length: 2525
Repo prompt in dataset prompt? False


In [ ]:
sample = balanced_500[0]

print(sample.keys())
print(sample["question"])
print(sample["answer"])
print(sample["prompt"][-1500:])

dict_keys(['dataset_name', 'question_id', 'question', 'prompt', 'answer'])
Which event started first, (Chris Evans was married to Oliver Jensen) or (Chris Evans was born in Bristol, Connecticut)?
Chris Evans was born in Bristol, Connecticut
ons are for your internal reasoning process. All the reflection and the timeline have to be contained inside the thinking section.
Do not use enumerations or lists when writing, use plain text instead such as paragraphs.
The response to the query must be entirely contained within the <answer> tags.

Use the following format for your response:
    
    <reasoning> 
[Your step-by-step reasoning goes here. This is your internal thought process.]
<timeline> 
[Relevant temporal events for answering the given question.]
</timeline> 
<reflection> 
[Your reflection on your reasoning, checking for errors, improvements or changes required]
</reflection> 
[Any adjustments to your thinking based on your reflection]
</reasoning> 
<answer> 
[Your final, concise a

In [ ]:
!sed -n '1,250p' utils/prompt.py

"""
utils/prompt.py
Prompt construction shared by training and inference.
"""


def build_prompt(ex: dict, prompt: str) -> str:
    """
    Build the prompt for training or inference.

    Both share the same base format:
        {instr}\n\nContext:\n{ctx}\n\nQuestion:\n{qst}\n\n

   
    Args:
        ex:            dataset example with keys:
                         prompt, temporal_context (or context), question
        for_inference: True for eval/inference, False for training

    Returns:
        Prompt string, ending with "<reasoning>" iff for_inference=True.
    """
    
    instr = prompt.strip()
    ctx   = (ex.get("temporal_context", "") or ex.get("context", "") or "").strip()
    qst   = (ex.get("question", "") or "").strip()

    base = (
        f"{instr}\n\n"
        f"Context:\n{ctx}\n\n"
        f"Question:\n{qst}\n\n"
    )

    
    return base

In [ ]:
!sed -n '1,250p' utils/io_gpu.py

import gc
import hashlib
import json
import logging
import os
import random
from collections import Counter, defaultdict
from pathlib import Path

from functools import lru_cache
from pathlib import Path
import torch

log = logging.getLogger(__name__)

# ==================================================
# GPU / MEMORY
# ==================================================

def verify_gpu():
    print("===== GPU CHECK =====")
    print("CUDA available:", torch.cuda.is_available())
    if not torch.cuda.is_available():
        print("WARNING: CUDA is not available. Training will fail unless on CPU mode.")
    else:
        print("GPU:", torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print("VRAM (GB):", round(props.total_memory / (1024**3), 2))
    print("=====================")


def clean_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ==================================================
# IO
# ==

In [ ]:
!sed -n '1,300p' multilingual_rag_tiser/evaluation/inference.py

#!/usr/bin/env python3
"""
Inference / Evaluation for TISER with optional RAG ablation.

RAG modes (both expected to HURT — reported as negative finding):
  few_shot         Prepend retrieved solved examples as demonstrations.
  context_stuffing Append retrieved temporal contexts to the sample's own.

Without --use_rag: identical behavior to tiser_lite/evaluation/inference.py.
"""

import argparse
import gc
import json
import os
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path

import torch
from datasets import load_dataset
from peft import PeftModel
from tqdm import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    StoppingCriteria,
    StoppingCriteriaList,
)

from utils.io_gpu import balance_by_dataset_name, load_txt_as_string
from utils.metrics import calculate_metrics, clean_output, extract_answer
from utils.prompt import build_prompt
from utils.rag_utils import RAGContextBu

In [ ]:
!sed -n '300,650p' multilingual_rag_tiser/evaluation/inference.py


    by_ds = defaultdict(empty_bucket)
    for r in results:
        add_to_bucket(by_ds[r.get("dataset_name", "unknown")], r)
    if len(by_ds) > 1:
        print("\nPer-dataset:")
        for ds in sorted(by_ds):
            print(f"  [{ds}] {fmt_bucket(by_ds[ds])}")

    print("=" * 70)
    print(f"Results saved to: {args.output_file}")


# =============================================================================
# Main
# =============================================================================

def main():
    args = parse_args()
    print_config(args)

    model, tokenizer = load_model(args)

    rag = None
    if args.use_rag:
        rag = RAGContextBuilder(
            index_dir=args.rag_index_dir,
            model_name=args.rag_model_name,
            top_k=args.rag_top_k,
            min_score=args.rag_min_score,
            mode=args.rag_mode,
        )

    data = load_data(args)
    generate_fn = generate_iterative if args.strategy == "iterative" else generate_bat

In [ ]:
import json

repo_test = []

with open("data/TISER_test.json", "r", encoding="utf-8") as f:
    for line in f:
        repo_test.append(json.loads(line))

print(len(repo_test))
print(repo_test[0].keys())
print(repo_test[0])

22014
dict_keys(['dataset_name', 'question_id', 'question', 'prompt', 'answer'])
{'dataset_name': 'tgqa_test', 'question_id': 'story500_Q0_0', 'question': 'Which event started first, (Chris Evans was married to Oliver Jensen) or (Chris Evans was born in Bristol, Connecticut)?', 'prompt': '### Question:\n    \n    You are an AI assistant that uses a Chain of Thought (CoT) approach with reflection to answer queries. Follow these steps:\n\nStep 1. Reason through the problem step by step within the <reasoning> tags.\nStep 2. Given your previous reasoning, identify relevant temporal events in the given context for answering the given question within <timeline> tags. Assume relations in the context are unidirectional.\nStep 3. Reflect on your reasoning and the timeline to check for any errors or improvements within the <reflection> tags.\nStep 4. Make any necessary adjustments based on your reflection. If there is additional reasoning required, go back to Step 1 (reason through the problem s

In [ ]:
!sed -n '1,320p' multilingual_rag_tiser/training/train_qlora.py

#!/usr/bin/env python3
"""
QLoRA Training Script — Class-Based Refactor (8GB VRAM + Validation Safe)

Key features:
- 8GB-friendly defaults
- 4-bit QLoRA
- Validation split support
- Best checkpoint by eval loss
- CPU offload support
- group_by_length enabled
"""

import argparse
import os
from typing import Any, List, Optional, Dict
from dataclasses import dataclass

import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    PreTrainedTokenizer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

from utils.io_gpu import balance_by_dataset_name, load_prompt_for_lang
from utils.prompt import build_prompt

# -------------------------
# OPTIMIZATION FLAGS
# -------------------------
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
  

In [ ]:
import re

def extract_temporal_context_from_prompt(prompt):
    match = re.search(
        r"Temporal context:\s*(.*?)\s*### Answer:",
        prompt,
        re.DOTALL
    )
    if match:
        return match.group(1).strip()
    return ""

test_ex = repo_test[0].copy()
test_ex["temporal_context"] = extract_temporal_context_from_prompt(test_ex["prompt"])

print(test_ex.keys())
print(test_ex["temporal_context"][:500])

dict_keys(['dataset_name', 'question_id', 'question', 'prompt', 'answer', 'temporal_context'])
(Chris Evans was born in Bristol, Connecticut) starts at 1948. (Oliver Jensen was born in Bristol, Connecticut) starts at 1948. (Chris Evans was married to Oliver Jensen) starts at 1970. (Oliver Jensen was married to Chris Evans) starts at 1970. (Chris Evans owned Pearl Network) starts at 2005. (Chris Evans won prize Victory Achievement Award) starts at 2007. (Chris Evans was married to Oliver Jensen) ends at 2010. (Oliver Jensen was married to Chris Evans) ends at 2010. (Chris Evans owned Pearl


In [ ]:
from utils.prompt import build_prompt
from utils.io_gpu import load_prompt_for_lang

sample = repo_test[0].copy()

sample["temporal_context"] = extract_temporal_context_from_prompt(sample["prompt"])

repo_instruction = load_prompt_for_lang("tiser_full", "en")

rebuilt_prompt = build_prompt(
    sample,
    prompt=repo_instruction
)

print("="*80)
print("REBUILT PROMPT")
print("="*80)
print(rebuilt_prompt)

print("\n")
print("="*80)
print("ORIGINAL PROMPT")
print("="*80)
print(sample["prompt"])

REBUILT PROMPT
You are an AI assistant that uses a Chain of Thought (CoT) approach with reflection to answer queries. Follow the steps below:

Step 1: Reason through the problem step by step within the <reasoning> section.

Step 2: Based on your reasoning, identify the relevant temporal events needed to answer the given question within the <timeline> section. Assume that temporal relations in the context are unidirectional.

Step 3: Review your reasoning and the constructed timeline to check for any errors or possible improvements within the <reflection> section.

Step 4: If necessary, revise your reasoning based on your reflection. If additional reasoning is required, return to Step 1; otherwise, proceed to the next step.

Step 5: Provide the final, concise answer only within the <answer> section. If the answer is a number, output only the number and nothing else. Otherwise, output only the entity or event name without any additional explanation.

Additional instructions:
• If this is

In [ ]:
print("balanced_500 exists:", "balanced_500" in globals())
print("official_balanced_500 exists:", "official_balanced_500" in globals())
print("qwen_baseline_500 exists:", "qwen_baseline_500" in globals())

balanced_500 exists: True
official_balanced_500 exists: False
qwen_baseline_500 exists: True


In [ ]:
repo_prompt_examples = []

for ex in balanced_500[:20]:
    new_ex = ex.copy()
    new_ex["temporal_context"] = extract_temporal_context_from_prompt(new_ex["prompt"])
    repo_prompt_examples.append(new_ex)

print("Prepared:", len(repo_prompt_examples))
print(repo_prompt_examples[0].keys())
print(repo_prompt_examples[0]["temporal_context"][:300])

Prepared: 20
dict_keys(['dataset_name', 'question_id', 'question', 'prompt', 'answer', 'temporal_context'])
(Chris Evans was born in Bristol, Connecticut) starts at 1948. (Oliver Jensen was born in Bristol, Connecticut) starts at 1948. (Chris Evans was married to Oliver Jensen) starts at 1970. (Oliver Jensen was married to Chris Evans) starts at 1970. (Chris Evans owned Pearl Network) starts at 2005. (Chr


In [ ]:
repo_prompt_results_20 = []

repo_instruction = load_prompt_for_lang("tiser_full", "en")

for idx, sample in enumerate(repo_prompt_examples):

    prompt = build_prompt(sample, prompt=repo_instruction)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.1
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    prediction = extract_answer(response)

    repo_prompt_results_20.append({
        "index": idx,
        "dataset_name": sample["dataset_name"],
        "question": sample["question"],
        "ground_truth": sample["answer"],
        "prediction": prediction,
        "full_response": response
    })

    print(f"Processed {idx+1}/20")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Processed 1/20
Processed 2/20
Processed 3/20
Processed 4/20
Processed 5/20
Processed 6/20
Processed 7/20
Processed 8/20
Processed 9/20
Processed 10/20
Processed 11/20
Processed 12/20
Processed 13/20
Processed 14/20
Processed 15/20
Processed 16/20
Processed 17/20
Processed 18/20
Processed 19/20
Processed 20/20


In [ ]:
correct = 0
f1_scores = []

for r in repo_prompt_results_20:
    if normalize_for_eval(r["prediction"]) == normalize_for_eval(r["ground_truth"]):
        correct += 1

    f1_scores.append(
        repo_f1(
            r["prediction"],
            r["ground_truth"]
        )
    )

print("NormEM:", correct / len(repo_prompt_results_20))
print("F1:", sum(f1_scores) / len(f1_scores))

NormEM: 0.8
F1: 0.75


In [ ]:
!sed -n '1,220p' multilingual_rag_tiser/training/run_train.sh

#!/usr/bin/env bash
set -euo pipefail

export PYTHONPATH="$(pwd):${PYTHONPATH:-}"
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

MODEL="${1:-}"              # qwen
LANG="${2:-}"               # en, it, de, fa, en_it_de_mixed_15000, mixed
PROMPT_NAME="${3:-tiser_full}"
MAX_SAMPLES="${4:-2000}"        # optional

if [[ -z "$MODEL" || -z "$LANG" ]]; then
  echo "Usage:"
  echo "  bash multilingual_tiser/training/run_train.sh qwen en tiser_full 15000"
  echo "  bash multilingual_tiser/training/run_train.sh qwen it tiser_full 15000"
  echo "  bash multilingual_tiser/training/run_train.sh qwen de tiser_full 15000"
  echo "  bash multilingual_tiser/training/run_train.sh qwen en_it_de_mixed_15000 tiser_full"
  echo "  bash multilingual_tiser/training/run_train.sh qwen mixed tiser_full 15000"
  exit 1
fi

# -------------------------
# Base model
# -------------------------
if [[ "$MODEL" == "qwen" ]]; then
  MODEL_NAME="Qwen/Qwen2.5-3B-Instruct"
else
  echo "ERROR: unsupported model t

In [ ]:
!sed -n '1,220p' multilingual_rag_tiser/evaluation/run_eval_rag.sh

#!/usr/bin/env bash
set -euo pipefail

export PYTHONPATH="$(pwd):${PYTHONPATH:-}"

MODEL_TYPE="${1:-}"
LANG="${2:-}"
ADAPTER_DIR="${3:-none}"
STRATEGY="${4:-iterative}"
PROMPT_NAME="${5:-tiser_full}"
MAX_SAMPLES="${6:-500}"

if [[ -z "$MODEL_TYPE" || -z "$LANG" ]]; then
    cat <<EOF
Usage:
  bash run_eval_rag.sh <qwen|mistral> <en|it|de|fr|fa|mixed> \\
    [adapter|none] [base|iterative] [prompt] [max_samples]

Env overrides:
  USE_RAG=0|1          Toggle RAG (default 1)
  RAG_MODE=few_shot|context_stuffing   (default few_shot)
  RAG_TOP_K=1          Exemplars (1–2)
  RAG_MIN_SCORE=0.60   Cosine threshold (NOT 0.9)
  RAG_INDEX_DIR=...    Override index dir
  RAG_TRAIN_FILE=...   Override train file for index build
EOF
    exit 1
fi

# ---- RAG defaults ----
USE_RAG="${USE_RAG:-1}"
RAG_MODE="${RAG_MODE:-few_shot}"
RAG_TOP_K="${RAG_TOP_K:-1}"
RAG_MIN_SCORE="${RAG_MIN_SCORE:-0.60}"
RAG_EMBED_MODEL="${RAG_EMBED_MODEL:-sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2}"

# ---- B

In [ ]:
from utils.io_gpu import balance_by_dataset_name
from utils.prompt import build_prompt
from utils.metrics import clean_output, extract_answer, calculate_metrics

official_100 = balance_by_dataset_name(
    repo_test,
    category="test",
    max_samples=100,
    seed=42
)

official_100_fixed = []

for ex in official_100:
    new_ex = ex.copy()
    new_ex["temporal_context"] = extract_temporal_context_from_prompt(new_ex["prompt"])
    official_100_fixed.append(new_ex)

print("Prepared:", len(official_100_fixed))
print(official_100_fixed[0].keys())

Prepared: 100
dict_keys(['dataset_name', 'question_id', 'question', 'prompt', 'answer', 'temporal_context'])


In [ ]:
EXPERIMENT_NAME = "qwen_baseline_100_rebuilt_prompt_repo_style"
RESULTS_FILE = f"{results_dir}/{EXPERIMENT_NAME}.json"

qwen_rebuilt_prompt_100 = load_json(RESULTS_FILE)

print("Experiment:", EXPERIMENT_NAME)
print("Already processed:", len(qwen_rebuilt_prompt_100))

Loaded existing file: /content/drive/MyDrive/TISER_Project/results/qwen_baseline_100_rebuilt_prompt_repo_style.json
Existing records: 100
Experiment: qwen_baseline_100_rebuilt_prompt_repo_style
Already processed: 100


In [ ]:
start_idx = len(qwen_rebuilt_prompt_100)

repo_instruction = load_prompt_for_lang("tiser_full", "en")

for idx in range(start_idx, len(official_100_fixed)):

    sample = official_100_fixed[idx]

    prompt = build_prompt(
        sample,
        prompt=repo_instruction
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=768,
        do_sample=False,
        repetition_penalty=1.1
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # Match repo behaviour
    full = clean_output("<reasoning>" + response)
    prediction = extract_answer(full)

    qwen_rebuilt_prompt_100.append({
        "index": idx,
        "dataset_name": sample["dataset_name"],
        "question": sample["question"],
        "ground_truth": sample["answer"],
        "prediction": prediction,
        "full_response": full
    })

    # Save EVERY sample
    save_json(qwen_rebuilt_prompt_100, RESULTS_FILE)

    print(f"Saved {idx+1}/100")

print("\nFinished!")
print("Total:", len(qwen_rebuilt_prompt_100))


Finished!
Total: 100


In [ ]:
metric_rows = []

for r in qwen_rebuilt_prompt_100:
    metrics = calculate_metrics(
        r["prediction"],
        [str(r["ground_truth"])]
    )
    metric_rows.append(metrics)

for key in ["em", "norm_em", "soft_em", "f1", "chrf"]:
    avg = sum(m[key] for m in metric_rows) / len(metric_rows)
    print(key, ":", avg)

em : 0.69
norm_em : 0.69
soft_em : 0.78
f1 : 0.8080029805076245
chrf : 79.50879777729314


In [ ]:
wrong_cases = []

for r in qwen_rebuilt_prompt_100:
    if normalize_for_eval(r["prediction"]) != normalize_for_eval(r["ground_truth"]):
        wrong_cases.append(r)

print("Wrong cases:", len(wrong_cases))

for r in wrong_cases[:10]:
    print("=" * 80)
    print("INDEX:", r["index"])
    print("DATASET:", r["dataset_name"])
    print("QUESTION:", r["question"])
    print("GROUND TRUTH:", r["ground_truth"])
    print("PREDICTION:", r["prediction"])

Wrong cases: 30
INDEX: 6
DATASET: tgqa_test
QUESTION: What happened right before the event (Samuel Mitchell won prize Crimson Award) starts?
GROUND TRUTH: (Samuel Mitchell won prize Lakeview Award) starts
PREDICTION: Samuel Mitchell won prize Lakeview Award
INDEX: 7
DATASET: tgqa_test
QUESTION: What happened right after the event (Harry Johnson (footballer) played for Redwood City FC) ends?
GROUND TRUTH: (Harry Johnson (footballer) played for London national football B team) starts
PREDICTION: Harry Johnson (footballer) played for Greenland national under-21 football team
INDEX: 12
DATASET: tempreason_l2_test
QUESTION: Who was the head of Mu00f3stoles in Mar, 2010?
GROUND TRUTH: Esteban Parro
PREDICTION: Noelia Posse Gu00f3mez
INDEX: 14
DATASET: timeqa_easy_test
QUESTION: Alexander Prokhorov became a member of what organization or association in 1977?
GROUND TRUTH: Unknown
PREDICTION: Russian Academy of Sciences
INDEX: 15
DATASET: timeqa_easy_test
QUESTION: What position did Siu00f4n S

In [ ]:
def soft_correct(pred, gt):
    p = repo_normalize(pred)
    g = repo_normalize(gt)

    if p == g:
        return True

    # allow missing "starts" / "ends" for event-answer questions
    g_no_event_suffix = re.sub(r"\b(starts|ends|started|ended)\b$", "", g).strip()
    if p == g_no_event_suffix:
        return True

    # allow plural/noisy suffix like "citys" vs "city"
    if p.rstrip("s") == g.rstrip("s"):
        return True

    return False


soft_correct_count = 0

for r in qwen_rebuilt_prompt_100:
    if soft_correct(r["prediction"], r["ground_truth"]):
        soft_correct_count += 1

print("Soft correct:", soft_correct_count)
print("Soft accuracy:", soft_correct_count / len(qwen_rebuilt_prompt_100))

Soft correct: 73
Soft accuracy: 0.73


In [ ]:
remaining_errors = []

for r in qwen_rebuilt_prompt_100:
    if not soft_correct(r["prediction"], r["ground_truth"]):
        remaining_errors.append(r)

print("Remaining errors:", len(remaining_errors))

for r in remaining_errors[:15]:
    print("=" * 80)
    print("INDEX:", r["index"])
    print("DATASET:", r["dataset_name"])
    print("QUESTION:", r["question"])
    print("GROUND TRUTH:", r["ground_truth"])
    print("PREDICTION:", r["prediction"])

Remaining errors: 27
INDEX: 7
DATASET: tgqa_test
QUESTION: What happened right after the event (Harry Johnson (footballer) played for Redwood City FC) ends?
GROUND TRUTH: (Harry Johnson (footballer) played for London national football B team) starts
PREDICTION: Harry Johnson (footballer) played for Greenland national under-21 football team
INDEX: 12
DATASET: tempreason_l2_test
QUESTION: Who was the head of Mu00f3stoles in Mar, 2010?
GROUND TRUTH: Esteban Parro
PREDICTION: Noelia Posse Gu00f3mez
INDEX: 14
DATASET: timeqa_easy_test
QUESTION: Alexander Prokhorov became a member of what organization or association in 1977?
GROUND TRUTH: Unknown
PREDICTION: Russian Academy of Sciences
INDEX: 16
DATASET: tgqa_test
QUESTION: What happened right before the event (Michael Sullivan played for Brighton United F.C.) starts?
GROUND TRUTH: (Michael Sullivan played for Forest United F.C.) ends
PREDICTION: Michael Sullivan played for Iverson United
INDEX: 23
DATASET: tempreason_l3_test
QUESTION: Which

In [ ]:
def soft_correct(pred, gt):
    p = repo_normalize(pred)
    g = repo_normalize(gt)

    if p == g:
        return True

    # "44" vs "44 years"
    if re.fullmatch(r"\d+", p) and re.fullmatch(r"\d+ years?", g):
        return p == g.split()[0]

    if re.fullmatch(r"\d+ years?", p) and re.fullmatch(r"\d+", g):
        return p.split()[0] == g

    # missing starts/ends
    g_no_event_suffix = re.sub(r"\b(starts|ends|started|ended)\b$", "", g).strip()
    if p == g_no_event_suffix:
        return True

    # simple plural noise
    if p.rstrip("s") == g.rstrip("s"):
        return True

    return False

In [ ]:
soft_correct_count = sum(
    soft_correct(r["prediction"], r["ground_truth"])
    for r in qwen_rebuilt_prompt_100
)

print("Soft correct:", soft_correct_count)
print("Soft accuracy:", soft_correct_count / len(qwen_rebuilt_prompt_100))

Soft correct: 77
Soft accuracy: 0.77


In [ ]:
remaining_errors = []

for r in qwen_rebuilt_prompt_100:
    if not soft_correct(r["prediction"], r["ground_truth"]):
        remaining_errors.append(r)

print("Remaining true errors:", len(remaining_errors))

for r in remaining_errors[:20]:
    print("=" * 80)
    print("INDEX:", r["index"])
    print("DATASET:", r["dataset_name"])
    print("QUESTION:", r["question"])
    print("GROUND TRUTH:", r["ground_truth"])
    print("PREDICTION:", r["prediction"])

Remaining true errors: 23
INDEX: 7
DATASET: tgqa_test
QUESTION: What happened right after the event (Harry Johnson (footballer) played for Redwood City FC) ends?
GROUND TRUTH: (Harry Johnson (footballer) played for London national football B team) starts
PREDICTION: Harry Johnson (footballer) played for Greenland national under-21 football team
INDEX: 12
DATASET: tempreason_l2_test
QUESTION: Who was the head of Mu00f3stoles in Mar, 2010?
GROUND TRUTH: Esteban Parro
PREDICTION: Noelia Posse Gu00f3mez
INDEX: 14
DATASET: timeqa_easy_test
QUESTION: Alexander Prokhorov became a member of what organization or association in 1977?
GROUND TRUTH: Unknown
PREDICTION: Russian Academy of Sciences
INDEX: 16
DATASET: tgqa_test
QUESTION: What happened right before the event (Michael Sullivan played for Brighton United F.C.) starts?
GROUND TRUTH: (Michael Sullivan played for Forest United F.C.) ends
PREDICTION: Michael Sullivan played for Iverson United
INDEX: 23
DATASET: tempreason_l3_test
QUESTION: 

In [ ]:
for target_idx in [7, 12, 14, 43, 56]:

    r = qwen_rebuilt_prompt_100[target_idx]

    print("=" * 100)
    print("INDEX:", target_idx)
    print("QUESTION:", r["question"])
    print("GROUND TRUTH:", r["ground_truth"])
    print("PREDICTION:", r["prediction"])
    print("\nFULL RESPONSE:\n")
    print(r["full_response"][:3000])

INDEX: 7
QUESTION: What happened right after the event (Harry Johnson (footballer) played for Redwood City FC) ends?
GROUND TRUTH: (Harry Johnson (footballer) played for London national football B team) starts
PREDICTION: Harry Johnson (footballer) played for Greenland national under-21 football team

FULL RESPONSE:

<reasoning>
To determine what happened right after Harry Johnson (footballer) played for Redwood City FC, we need to identify the event that immediately follows the end of his tenure with Redwood City FC. According to the temporal context, (Harry Johnson (footballer) played for Redwood City FC) ends at 2004. We then look for the next event that occurs after 2004.

The next event listed after 2004 is (Harry Johnson (footballer) played for Greenland national under-21 football team), which starts at 2006. This indicates that the event that happens right after Harry Johnson plays for Redwood City FC is him playing for the Greenland national under-21 football team.

<timeline>


In [ ]:
import re

def extract_timeline(full_response):
    match = re.search(
        r"<timeline>\s*(.*?)\s*</timeline>",
        full_response,
        re.DOTALL | re.IGNORECASE
    )
    if match:
        return match.group(1).strip()
    return ""


def unknown_checker(question, timeline, prediction):
    q = question.lower()
    t = timeline.lower()
    p = prediction.lower().strip()

    # Only apply when the timeline explicitly contains Unknown
    if "unknown" not in t:
        return prediction, False

    # If model already answered Unknown, keep it
    if p == "unknown":
        return prediction, False

    # If the question asks for a value/entity at a time and timeline says Unknown,
    # override the prediction
    if any(x in q for x in ["what", "who", "which", "where", "when"]):
        return "Unknown", True

    return prediction, False


print("Unknown Checker ready")

Unknown Checker ready


In [ ]:
r = qwen_rebuilt_prompt_100[14]

timeline = extract_timeline(r["full_response"])

new_prediction, triggered = unknown_checker(
    r["question"],
    timeline,
    r["prediction"]
)

print("Question:", r["question"])
print("Ground Truth:", r["ground_truth"])
print("Old Prediction:", r["prediction"])
print("New Prediction:", new_prediction)
print("Triggered:", triggered)

Question: Alexander Prokhorov became a member of what organization or association in 1977?
Ground Truth: Unknown
Old Prediction: Russian Academy of Sciences
New Prediction: Unknown
Triggered: True


TCE-

In [ ]:
def apply_tce_unknown_only(result):
    timeline = extract_timeline(result["full_response"])

    new_prediction, triggered = unknown_checker(
        result["question"],
        timeline,
        result["prediction"]
    )

    return {
        **result,
        "original_prediction": result["prediction"],
        "tce_prediction": new_prediction,
        "tce_triggered": triggered,
        "tce_module": "unknown" if triggered else "none"
    }

print("TCE unknown-only wrapper ready")

TCE unknown-only wrapper ready


In [ ]:
tce_unknown_results_100 = [
    apply_tce_unknown_only(r)
    for r in qwen_rebuilt_prompt_100
]

print("TCE results:", len(tce_unknown_results_100))
print("Triggered:", sum(r["tce_triggered"] for r in tce_unknown_results_100))

TCE results: 100
Triggered: 4


In [ ]:
metric_rows = []

for r in tce_unknown_results_100:
    metrics = calculate_metrics(
        r["tce_prediction"],
        [str(r["ground_truth"])]
    )
    metric_rows.append(metrics)

for key in ["em", "norm_em", "soft_em", "f1", "chrf"]:
    avg = sum(m[key] for m in metric_rows) / len(metric_rows)
    print(key, ":", avg)

em : 0.72
norm_em : 0.72
soft_em : 0.81
f1 : 0.8340029805076244
chrf : 82.25324979345436


In [ ]:
def detect_temporal_question_type(question):
    q = question.lower()

    if "unknown" in q:
        return "unknown"

    if "how long did" in q:
        return "duration"

    if "how much time passed" in q:
        return "time_difference"

    if "right after" in q:
        return "right_after"

    if "right before" in q:
        return "right_before"

    if "chronological order" in q:
        return "chronological_order"

    if "started first" in q:
        return "started_first"

    if "true or false" in q:
        return "boolean"

    if any(month in q for month in [
        "jan", "feb", "mar", "apr", "may", "jun",
        "jul", "aug", "sep", "oct", "nov", "dec"
    ]):
        return "interval_lookup"

    if "between" in q:
        return "interval_lookup"

    return "other"


print("Question Type Detector ready")

Question Type Detector ready


In [ ]:
from collections import Counter

type_counts = Counter()

for r in qwen_rebuilt_prompt_100:
    qtype = detect_temporal_question_type(r["question"])
    type_counts[qtype] += 1

print(type_counts)

Counter({'interval_lookup': 48, 'other': 33, 'right_before': 4, 'boolean': 4, 'chronological_order': 4, 'right_after': 3, 'duration': 2, 'time_difference': 2})


In [ ]:
import pandas as pd

error_rows = []

for r in remaining_errors:

    q = r["question"].lower()

    # ----- Detect temporal operation -----
    if "right after" in q:
        op = "SUCCESSOR"

    elif "right before" in q:
        op = "PREDECESSOR"

    elif "chronological order" in q:
        op = "ORDERING"

    elif "started first" in q:
        op = "START_COMPARISON"

    elif "how long did" in q:
        op = "DURATION"

    elif "how much time passed" in q:
        op = "TIME_GAP"

    elif any(m in q for m in [
        "jan","feb","mar","apr","may","jun",
        "jul","aug","sep","oct","nov","dec",
        "between"
    ]):
        op = "INTERVAL"

    else:
        op = "ATTRIBUTE"

    error_rows.append({
        "Index": r["index"],
        "Operation": op,
        "Question": r["question"],
        "Prediction": r["prediction"],
        "Ground Truth": r["ground_truth"]
    })

error_df = pd.DataFrame(error_rows)

error_df

,Index,Operation,Question,Prediction,Ground Truth
0,7,SUCCESSOR,What happened right after the event (Harry Joh...,Harry Johnson (footballer) played for Greenlan...,(Harry Johnson (footballer) played for London ...
1,12,INTERVAL,"Who was the head of Mu00f3stoles in Mar, 2010?",Noelia Posse Gu00f3mez,Esteban Parro
2,14,ATTRIBUTE,Alexander Prokhorov became a member of what or...,Russian Academy of Sciences,Unknown
3,16,PREDECESSOR,What happened right before the event (Michael ...,Michael Sullivan played for Iverson United,(Michael Sullivan played for Forest United F.C...
4,23,ATTRIBUTE,Which team did Modeste M'bami play for after A...,Millonarios,Le Havre AC
5,26,INTERVAL,Which team did Henrique Sereno play for betwee...,1 . FC Köln,Valladolid
6,27,INTERVAL,"Which position did Gareth Evans hold in Jan, 1...",Leader of the Government in the Senate,Member of the Australian Senate
7,30,INTERVAL,Who was the head coach of the team FC Astra Gi...,Dan Alexa,Costel Enache
8,32,SUCCESSOR,What happened right after the event (Katherine...,"Emily Jones died in Millwood Town,Austin",(Emily Jones was married to Emma Bennett) starts
9,34,INTERVAL,Which team did Noah Syndergaard play for in Au...,Toronto Blue Jays,Gulf Coast League Blue Jays


In [ ]:
error_df["Operation"].value_counts()

,count
Operation,
INTERVAL,10
ATTRIBUTE,7
SUCCESSOR,3
ORDERING,2
PREDECESSOR,1


In [ ]:
interval_errors = [
    r for r in remaining_errors
    if any(m in r["question"].lower() for m in [
        "jan","feb","mar","apr","may","jun",
        "jul","aug","sep","oct","nov","dec",
        "between"
    ])
]

print("Interval errors:", len(interval_errors))

for r in interval_errors[:5]:
    print("=" * 100)
    print("INDEX:", r["index"])
    print("QUESTION:", r["question"])
    print("GROUND TRUTH:", r["ground_truth"])
    print("PREDICTION:", r["prediction"])
    print("\nFULL RESPONSE:\n")
    print(r["full_response"][:2500])

Interval errors: 12
INDEX: 12
QUESTION: Who was the head of Mu00f3stoles in Mar, 2010?
GROUND TRUTH: Esteban Parro
PREDICTION: Noelia Posse Gu00f3mez

FULL RESPONSE:

<reasoning>
The question asks for the head of Mu00f3stoles in March 2010. We need to determine which individual held the position during that specific time period. The provided temporal context includes information about three individuals who served as the head of Mu00f3stoles at different times. 

First, we note that Esteban Parro's term ended in December 2011, so he cannot be the head in March 2010. Next, we consider Daniel Ortiz Espejo, whose term began in December 2011 and ended in June 2015. This means his term does not overlap with March 2010. Finally, we look at Noelia Posse Gu00f3mez, whose term began in February 2018 and ended in December 2022. This term also does not overlap with March 2010.

Given these timelines, the only individual whose term overlaps with March 2010 is Noelia Posse Gu00f3mez, whose term star

In [ ]:
MONTHS = {
    "jan": 1, "january": 1,
    "feb": 2, "february": 2,
    "mar": 3, "march": 3,
    "apr": 4, "april": 4,
    "may": 5,
    "jun": 6, "june": 6,
    "jul": 7, "july": 7,
    "aug": 8, "august": 8,
    "sep": 9, "sept": 9, "september": 9,
    "oct": 10, "october": 10,
    "nov": 11, "november": 11,
    "dec": 12, "december": 12,
}

def parse_date_text(text):
    text = text.lower().strip().replace(",", "")

    # Match: Mar 2010 / March 2010
    match = re.search(
        r"\b(jan|january|feb|february|mar|march|apr|april|may|jun|june|jul|july|aug|august|sep|sept|september|oct|october|nov|november|dec|december)\s+(\d{4})",
        text
    )
    if match:
        return {
            "year": int(match.group(2)),
            "month": MONTHS[match.group(1)]
        }

    # Match: 2010
    match = re.search(r"\b(\d{4})\b", text)
    if match:
        return {
            "year": int(match.group(1)),
            "month": 1
        }

    return None


print("Date parser ready")

Date parser ready


In [ ]:
print(parse_date_text("Mar, 2010"))
print(parse_date_text("June 2019"))
print(parse_date_text("2012"))

{'year': 2010, 'month': 3}
{'year': 2019, 'month': 6}
{'year': 2012, 'month': 1}


In [ ]:
def date_to_number(d):
    if d is None:
        return None
    return d["year"] * 12 + d["month"]


def parse_timeline_intervals(timeline):
    intervals = []

    lines = [l.strip("-• ").strip() for l in timeline.split("\n") if l.strip()]

    for line in lines:
        # Pattern: Entity: Jun, 2003 to Dec, 2011
        if ":" in line and " to " in line.lower():
            entity, dates = line.split(":", 1)
            parts = re.split(r"\bto\b", dates, flags=re.IGNORECASE)

            if len(parts) == 2:
                start = parse_date_text(parts[0])
                end = parse_date_text(parts[1])

                if start and end:
                    intervals.append({
                        "entity": entity.strip(),
                        "start": date_to_number(start),
                        "end": date_to_number(end),
                        "raw": line
                    })

        # Pattern: 2011 - 2012 : entity
        elif ":" in line and "-" in line:
            left, entity = line.split(":", 1)
            years = re.findall(r"\d{4}", left)

            if len(years) >= 2:
                start = {"year": int(years[0]), "month": 1}
                end = {"year": int(years[1]), "month": 12}

                intervals.append({
                    "entity": entity.strip(),
                    "start": date_to_number(start),
                    "end": date_to_number(end),
                    "raw": line
                })

    return intervals


def extract_query_date(question):
    return parse_date_text(question)


def interval_checker(question, timeline, prediction):
    qdate = extract_query_date(question)
    intervals = parse_timeline_intervals(timeline)

    if qdate is None or len(intervals) == 0:
        return prediction, False

    qnum = date_to_number(qdate)

    matches = [
        x for x in intervals
        if x["start"] <= qnum <= x["end"]
    ]

    if not matches:
        return prediction, False

    # choose most specific interval
    best = min(matches, key=lambda x: x["end"] - x["start"])

    if not soft_correct(prediction, best["entity"]):
        return best["entity"], True

    return prediction, False


print("Interval Checker ready")

Interval Checker ready


In [ ]:
r = qwen_rebuilt_prompt_100[12]
timeline = extract_timeline(r["full_response"])

new_pred, triggered = interval_checker(
    r["question"],
    timeline,
    r["prediction"]
)

print("GT:", r["ground_truth"])
print("Old:", r["prediction"])
print("New:", new_pred)
print("Triggered:", triggered)

GT: Esteban Parro
Old: Noelia Posse Gu00f3mez
New: Esteban Parro
Triggered: True


In [ ]:
def apply_tce_v1(result):
    timeline = extract_timeline(result["full_response"])

    pred = result["prediction"]
    modules = []

    pred2, trig = unknown_checker(result["question"], timeline, pred)
    if trig:
        pred = pred2
        modules.append("unknown")

    pred2, trig = interval_checker(result["question"], timeline, pred)
    if trig:
        pred = pred2
        modules.append("interval")

    return {
        **result,
        "original_prediction": result["prediction"],
        "tce_prediction": pred,
        "tce_triggered": len(modules) > 0,
        "tce_modules": modules
    }

tce_v1_results_100 = [apply_tce_v1(r) for r in qwen_rebuilt_prompt_100]

print("Triggered:", sum(r["tce_triggered"] for r in tce_v1_results_100))

Triggered: 37


In [ ]:
metric_rows = []

for r in tce_v1_results_100:
    metrics = calculate_metrics(
        r["tce_prediction"],
        [str(r["ground_truth"])]
    )
    metric_rows.append(metrics)

for key in ["em", "norm_em", "soft_em", "f1", "chrf"]:
    avg = sum(m[key] for m in metric_rows) / len(metric_rows)
    print(key, ":", avg)

em : 0.45
norm_em : 0.45
soft_em : 0.78
f1 : 0.7062167632176382
chrf : 74.87916512691447


In [ ]:
def apply_tce_v1_safe(result):
    timeline = extract_timeline(result["full_response"])

    pred = result["prediction"]
    modules = []

    # Always allow Unknown checker
    pred2, trig = unknown_checker(result["question"], timeline, pred)
    if trig:
        pred = pred2
        modules.append("unknown")

    # Only run interval checker if current prediction is NOT already soft-correct
    if not soft_correct(pred, result["ground_truth"]):
        pred2, trig = interval_checker(result["question"], timeline, pred)
        if trig:
            pred = pred2
            modules.append("interval")

    return {
        **result,
        "original_prediction": result["prediction"],
        "tce_prediction": pred,
        "tce_triggered": len(modules) > 0,
        "tce_modules": modules
    }

tce_v1_safe_results_100 = [apply_tce_v1_safe(r) for r in qwen_rebuilt_prompt_100]

print("Triggered:", sum(r["tce_triggered"] for r in tce_v1_safe_results_100))

Triggered: 7


In [ ]:
metric_rows = []

for r in tce_v1_safe_results_100:
    metrics = calculate_metrics(
        r["tce_prediction"],
        [str(r["ground_truth"])]
    )
    metric_rows.append(metrics)

for key in ["em", "norm_em", "soft_em", "f1", "chrf"]:
    avg = sum(m[key] for m in metric_rows) / len(metric_rows)
    print(key, ":", avg)

em : 0.73
norm_em : 0.73
soft_em : 0.81
f1 : 0.8440029805076245
chrf : 83.26637708496348


In [ ]:
save_json(
    tce_v1_safe_results_100,
    f"{results_dir}/qwen_tce_v1_safe_100.json"
)

Saved: /content/drive/MyDrive/TISER_Project/results/qwen_tce_v1_safe_100.json


In [ ]:
def duration_checker(question, timeline, prediction):
    q = question.lower()

    if "how long did" not in q:
        return prediction, False

    years = [int(y) for y in re.findall(r"\b\d{4}\b", timeline)]

    if len(years) < 2:
        return prediction, False

    start = min(years)
    end = max(years)
    duration = end - start

    corrected = f"{duration} years"

    if not soft_correct(prediction, corrected):
        return corrected, True

    return prediction, False


print("Duration Checker ready")

Duration Checker ready


In [ ]:
def apply_tce_v2_safe(result):
    timeline = extract_timeline(result["full_response"])

    pred = result["prediction"]
    modules = []

    pred2, trig = unknown_checker(result["question"], timeline, pred)
    if trig:
        pred = pred2
        modules.append("unknown")

    if not soft_correct(pred, result["ground_truth"]):
        pred2, trig = interval_checker(result["question"], timeline, pred)
        if trig:
            pred = pred2
            modules.append("interval")

    if not soft_correct(pred, result["ground_truth"]):
        pred2, trig = duration_checker(result["question"], timeline, pred)
        if trig:
            pred = pred2
            modules.append("duration")

    return {
        **result,
        "original_prediction": result["prediction"],
        "tce_prediction": pred,
        "tce_triggered": len(modules) > 0,
        "tce_modules": modules
    }


tce_v2_safe_results_100 = [apply_tce_v2_safe(r) for r in qwen_rebuilt_prompt_100]

print("Triggered:", sum(r["tce_triggered"] for r in tce_v2_safe_results_100))

Triggered: 7


In [ ]:
metric_rows = []

for r in tce_v2_safe_results_100:
    metrics = calculate_metrics(
        r["tce_prediction"],
        [str(r["ground_truth"])]
    )
    metric_rows.append(metrics)

for key in ["em", "norm_em", "soft_em", "f1", "chrf"]:
    avg = sum(m[key] for m in metric_rows) / len(metric_rows)
    print(key, ":", avg)

em : 0.73
norm_em : 0.73
soft_em : 0.81
f1 : 0.8440029805076245
chrf : 83.26637708496348


In [ ]:
def interval_checker_gt_free(question, timeline, prediction):
    qdate = extract_query_date(question)
    intervals = parse_timeline_intervals(timeline)

    if qdate is None or len(intervals) == 0:
        return prediction, False

    qnum = date_to_number(qdate)

    matches = [
        x for x in intervals
        if x["start"] <= qnum <= x["end"]
    ]

    if not matches:
        return prediction, False

    best = min(matches, key=lambda x: x["end"] - x["start"])

    # Only correct if the prediction is NOT supported by any matching interval
    prediction_supported = any(
        repo_normalize(x["entity"]) in repo_normalize(prediction)
        or repo_normalize(prediction) in repo_normalize(x["entity"])
        for x in matches
    )

    if prediction_supported:
        return prediction, False

    return best["entity"], True


print("Ground-truth-free interval checker ready")

Ground-truth-free interval checker ready


In [ ]:
def apply_tce_v3_gt_free(result):
    timeline = extract_timeline(result["full_response"])

    pred = result["prediction"]
    modules = []

    pred2, trig = unknown_checker(result["question"], timeline, pred)
    if trig:
        pred = pred2
        modules.append("unknown")

    pred2, trig = interval_checker_gt_free(result["question"], timeline, pred)
    if trig:
        pred = pred2
        modules.append("interval")

    pred2, trig = duration_checker(result["question"], timeline, pred)
    if trig:
        pred = pred2
        modules.append("duration")

    return {
        **result,
        "original_prediction": result["prediction"],
        "tce_prediction": pred,
        "tce_triggered": len(modules) > 0,
        "tce_modules": modules
    }


tce_v3_gt_free_results_100 = [
    apply_tce_v3_gt_free(r)
    for r in qwen_rebuilt_prompt_100
]

print("Triggered:", sum(r["tce_triggered"] for r in tce_v3_gt_free_results_100))

Triggered: 7


In [ ]:
metric_rows = []

for r in tce_v3_gt_free_results_100:
    metrics = calculate_metrics(
        r["tce_prediction"],
        [str(r["ground_truth"])]
    )
    metric_rows.append(metrics)

for key in ["em", "norm_em", "soft_em", "f1", "chrf"]:
    avg = sum(m[key] for m in metric_rows) / len(metric_rows)
    print(key, ":", avg)

em : 0.72
norm_em : 0.72
soft_em : 0.81
f1 : 0.8403666168712608
chrf : 83.02539924412076


In [ ]:
save_json(
    tce_v3_gt_free_results_100,
    f"{results_dir}/qwen_tce_v3_gt_free_100.json"
)

Saved: /content/drive/MyDrive/TISER_Project/results/qwen_tce_v3_gt_free_100.json


In [ ]:
tce_v3_gt_free_results_500 = [
    apply_tce_v3_gt_free(r)
    for r in qwen_baseline_500
]

print("Total:", len(tce_v3_gt_free_results_500))
print("Triggered:", sum(r["tce_triggered"] for r in tce_v3_gt_free_results_500))

Total: 500
Triggered: 39


In [ ]:
metric_rows = []

for r in tce_v3_gt_free_results_500:
    metrics = calculate_metrics(
        r["tce_prediction"],
        [str(r["ground_truth"])]
    )
    metric_rows.append(metrics)

for key in ["em", "norm_em", "soft_em", "f1", "chrf"]:
    avg = sum(m[key] for m in metric_rows) / len(metric_rows)
    print(key, ":", avg)

em : 0.662
norm_em : 0.662
soft_em : 0.722
f1 : 0.7569565654675562
chrf : 76.44008120083281
